## Implementing a Text Generator Using Recurrent Neural Networks (RNNs)
In this section, we create a character-based text generator using Recurrent Neural Network (RNN) in TensorFlow and Keras. We'll implement an RNN that learns patterns from a text sequence to generate new text character-by-character.

In [64]:
import numpy as np
import torch
import torch.nn as nn

### <br>2. Defining the Input Text and Prepare Character Set
We define the input text and identify unique characters in the text which we'll encode for our model.

In [65]:
text = """Once more the storm is howling, and half hid
Under this cradle-hood and coverlid
My child sleeps on. There is no obstacle
But Gregory's wood and one bare hill
Whereby the haystack- and roof-levelling wind,
Bred on the Atlantic, can be stayed;
And for an hour I have walked and prayed
Because of the great gloom that is in my mind.

I have walked and prayed for this young child an hour
And heard the sea-wind scream upon the tower,
And under the arches of the bridge, and scream
In the elms above the flooded stream;
Imagining in excited reverie
That the future years had come,
Dancing to a frenzied drum,
Out of the murderous innocence of the sea.

May she be granted beauty and yet not
Beauty to make a stranger's eye distraught,
Or hers before a looking-glass, for such,
Being made beautiful overmuch,
Consider beauty a sufficient end,
Lose natural kindness and maybe
The heart-revealing intimacy
That chooses right, and never find a friend.

Helen being chosen found life flat and dull
And later had much trouble from a fool,
While that great Queen, that rose out of the spray,
Being fatherless could have her way
Yet chose a bandy-leggèd smith for man.
It's certain that fine women eat
A crazy salad with their meat
Whereby the Horn of Plenty is undone.

In courtesy I'd have her chiefly learned;
Hearts are not had as a gift but hearts are earned
By those that are not entirely beautiful;
Yet many, that have played the fool
For beauty's very self, has charm made wise,
And many a poor man that has roved,
Loved and thought himself beloved,
From a glad kindness cannot take his eyes.

May she become a flourishing hidden tree
That all her thoughts may like the linnet be,
And have no business but dispensing round
Their magnanimities of sound,
Nor but in merriment begin a chase,
Nor but in merriment a quarrel.
O may she live like some green laurel
Rooted in one dear perpetual place.

My mind, because the minds that I have loved,
The sort of beauty that I have approved,
Prosper but little, has dried up of late,
Yet knows that to be choked with hate
May well be of all evil chances chief.
If there's no hatred in a mind
Assault and battery of the wind
Can never tear the linnet from the leaf.

An intellectual hatred is the worst,
So let her think opinions are accursed.
Have I not seen the loveliest woman born
Out of the mouth of Plenty's horn,
Because of her opinionated mind
Barter that horn and every good
By quiet natures understood
For an old bellows full of angry wind?

Considering that, all hatred driven hence,
The soul recovers radical innocence
And learns at last that it is self-delighting,
Self-appeasing, self-affrighting,
And that its own sweet will is Heaven's will;
She can, though every face should scowl
And every windy quarter howl
Or every bellows burst, be happy still.

And may her bridegroom bring her to a house
Where all's accustomed, ceremonious;
For arrogance and hatred are the wares
Peddled in the thoroughfares.
How but in custom and in ceremony
Are innocence and beauty born?
Ceremony's a name for the rich horn,
And custom for the spreading laurel tree."""

chars = sorted(list(set(text)))  # why set then list then sort?

char_to_index = {char: i for i, char in enumerate(chars)}
index_to_char = {i: char for i, char in enumerate(chars)}

### Quick explanation of the building blocks above

- **`set(text)`** — removes duplicate characters, leaving only the *unique* characters (order is not guaranteed).
  ```python
  set("aab")  # {'a', 'b'}
  ```

- **`list(...)`** — converts that set into a list so it can be sorted and indexed.
  ```python
  list({'a', 'b'})  # ['a', 'b']
  ```

- **`sorted(...)`** — sorts the list. For characters this sorts by Unicode code point (e.g. space and uppercase letters come before lowercase letters).
  ```python
  sorted(['b', 'a', ' '])  # [' ', 'a', 'b']
  ```

- **`enumerate(chars)`** — walks through `chars` and pairs each item with its position (index), producing `(0, chars[0]), (1, chars[1]), ...`
  ```python
  list(enumerate(['a', 'b', 'c']))  # [(0, 'a'), (1, 'b'), (2, 'c')]
  ```

- **`char_to_index = {char: i for i, char in enumerate(chars)}`** — a *dict comprehension* that builds a lookup table mapping each character to the integer position it has in `chars`.
  ```python
  chars = [' ', 'G', 'T', 'a']
  char_to_index = {char: i for i, char in enumerate(chars)}
  # {' ': 0, 'G': 1, 'T': 2, 'a': 3}
  ```

This gives every unique character a unique integer ID. `index_to_char` does the reverse mapping (ID → character), which we'll need to turn the model's numeric predictions back into text.

### Why do we do this?

Neural networks only understand **numbers**, not letters. So before we can feed text into our RNN, we need a way to convert characters ↔ numbers:

- **`char_to_index`** — lets us **encode** the input text into a sequence of integers (numbers) that the model can process.
- **`index_to_char`** — lets us **decode** the model's numeric output (predicted IDs) back into readable characters/text.

In short: encode text → numbers to train the model, decode numbers → text to read what it generates.

### <br>3. Creating Sequences and Labels
To train the RNN, we need sequences of fixed length (seq_length) and the character following each sequence as the label.

In [66]:
seq_length = 10
sequences = []
labels = []

for i in range(len(text) - seq_length):
    seq = text[i:i + seq_length]
    label = text[i + seq_length]
    sequences.append([char_to_index[char] for char in seq])
    labels.append(char_to_index[label])

print(sequences[:5])
print(labels[:5])

X = np.array(sequences)
y = np.array(labels)

[[19, 40, 30, 32, 1, 39, 41, 44, 32, 1], [40, 30, 32, 1, 39, 41, 44, 32, 1, 46], [30, 32, 1, 39, 41, 44, 32, 1, 46, 35], [32, 1, 39, 41, 44, 32, 1, 46, 35, 32], [1, 39, 41, 44, 32, 1, 46, 35, 32, 1]]
[46, 35, 32, 1, 45]


### What this code does

We're building the **training data** for the RNN using a sliding window over `text`.

- **`seq_length = 3`** — each input example is 3 characters long, and the model's job is to predict the **next** (4th) character.

- **The loop** slides the window one character at a time across `text`:
  - `seq = text[i:i+seq_length]` → 3 consecutive characters (the **input**)
  - `label = text[i+seq_length]` → the character right after them (the **answer**)

  Example with `text = "This is..."`:
  - `i=0` → `seq = "Thi"`, `label = "s"`
  - `i=1` → `seq = "his"`, `label = " "`
  - `i=2` → `seq = "is "`, `label = "i"`

- **`[char_to_index[char] for char in seq]`** — converts each character in `seq` to its integer ID (e.g. `"Thi"` → `[2, 7, 8]`) and appends it to `sequences`.

- **`char_to_index[label]`** — converts the target character to its integer ID and appends it to `labels`.

- **`X = np.array(sequences)`** / **`y = np.array(labels)`** — convert the Python lists into NumPy arrays:
  - `X` shape: `(num_samples, seq_length)` — the inputs
  - `y` shape: `(num_samples,)` — the correct next character for each input

In short: for every 3-character chunk of text, we record "given these 3 characters, the next one is ___" — that's the pattern the RNN learns to predict.

### <br>4. Converting Sequences and Labels to One-Hot Encoding
For training we convert X and y into one-hot encoded tensors.

In [67]:
X_tensor = torch.tensor(X, dtype=torch.long)
y_tensor = torch.tensor(y, dtype=torch.long)

X_one_hot = torch.nn.functional.one_hot(X_tensor, num_classes=len(chars)).float()
y_one_hot = torch.nn.functional.one_hot(y_tensor, num_classes=len(chars)).float()

### Why do we need one-hot encoding?

`char_to_index` gives each character a number (e.g. `'a' → 3`), but plain integers imply an **order/magnitude** that doesn't actually exist between characters — the model would wrongly think `'d'` (3) is "more" than `'a'` (0), or that `'b'` is "between" `'a'` and `'c'`.

**One-hot encoding** fixes this by representing each character as a vector of length `len(chars)`, with a `1` at that character's index and `0`s everywhere else — so every character is treated as an equally distinct, independent category.

Example, if `len(chars) = 14` and `char_to_index['a'] = 3`:
```
'a' → [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
```

This is also the input/output shape the RNN expects: at each time step, the input is a vector of size `len(chars)` (one slot per possible character), and the output is a probability distribution over `len(chars)` possible "next characters".

### <br>5. Building the RNN Model
We create a simple RNN model with a hidden layer of 50 units and a Dense output layer with softmax activation.

In [68]:
class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)  # tanh by default, 1 layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]  # take the output of the last time step
        out = self.fc(out)
        return torch.softmax(out, dim=1)

model = CharRNN(input_size=len(chars), hidden_size=128, output_size=len(chars))
model

CharRNN(
  (rnn): RNN(54, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=54, bias=True)
)

<div align="center">

<table style="border:none; text-align:center; font-family:sans-serif; border-collapse:collapse;">
<tr>
<td></td>
<td style="background:#c8f7c5; border:2px solid #2ca02c; border-radius:8px; padding:8px 14px; font-size:15px;"><i>out₁ = h₁</i></td>
<td></td>
<td style="background:#c8f7c5; border:2px solid #2ca02c; border-radius:8px; padding:8px 14px; font-size:15px;"><i>out₂ = h₂</i></td>
<td></td>
<td style="background:#7be07b; border:2px solid #2ca02c; border-radius:8px; padding:8px 14px; font-size:15px;"><i>out₃ = h₃</i></td>
<td></td>
</tr>
<tr style="font-size:22px; color:#2ca02c;">
<td></td><td>↑</td><td></td><td>↑</td><td></td><td>↑</td><td></td>
</tr>
<tr>
<td style="font-size:13px; color:#a85a00; white-space:nowrap;">h₀ (zeros) →</td>
<td style="background:#ffd9a0; border:2px solid #e07b00; border-radius:8px; padding:16px 18px; font-size:16px; font-weight:bold;">RNN<br>cell</td>
<td style="font-size:13px; color:#a85a00; white-space:nowrap;">→ h₁ →</td>
<td style="background:#ffd9a0; border:2px solid #e07b00; border-radius:8px; padding:16px 18px; font-size:16px; font-weight:bold;">RNN<br>cell</td>
<td style="font-size:13px; color:#a85a00; white-space:nowrap;">→ h₂ →</td>
<td style="background:#ffd9a0; border:2px solid #e07b00; border-radius:8px; padding:16px 18px; font-size:16px; font-weight:bold;">RNN<br>cell</td>
<td style="font-size:13px; color:#a85a00; white-space:nowrap;">→ h₃</td>
</tr>
<tr style="font-size:22px; color:#1f77b4;">
<td></td><td>↑</td><td></td><td>↑</td><td></td><td>↑</td><td></td>
</tr>
<tr>
<td></td>
<td style="background:#cfe8ff; border:2px solid #1f77b4; border-radius:8px; padding:8px 14px; font-size:15px;"><i>x₁</i></td>
<td></td>
<td style="background:#cfe8ff; border:2px solid #1f77b4; border-radius:8px; padding:8px 14px; font-size:15px;"><i>x₂</i></td>
<td></td>
<td style="background:#cfe8ff; border:2px solid #1f77b4; border-radius:8px; padding:8px 14px; font-size:15px;"><i>x₃</i></td>
<td></td>
</tr>
</table>

<p style="font-size:12px; color:#2ca02c; margin-top:10px;">
<b>out[:, -1, :] = out₃ = h₃</b> → fed into <code>self.fc</code> → <code>softmax</code> → next-character probabilities
</p>

</div>

Think of the RNN as reading your sequence (e.g. `"Thi"`) **one character at a time**, keeping a running "summary" in its head as it goes.

- **`hidden_size = 50`** — that "summary" is just a list of **50 numbers**. That's the RNN's entire memory.

- **Reading `"Thi"` step by step:**
  1. Read `'T'` → update the summary → call it `h1`
  2. Read `'h'` (starting from `h1`) → update the summary → `h2`
  3. Read `'i'` (starting from `h2`) → update the summary → `h3`

- **`out, _ = self.rnn(x)`** does all 3 steps at once and hands back:
  - `out` = **every** summary along the way: `[h1, h2, h3]`
  - `_` = just the **last** one, `h3` — already included in `out`, so we ignore it

- **`out[:, -1, :]`** = grab `h3`, the summary *after reading the whole sequence*. This is the single thing we use to guess the next character.

- **`batch_first=True`** is just about ordering numbers in the shape — it means "tell me **how many sequences** first", e.g. `(32 sequences, 3 characters each, 50-number summary)` instead of `(3, 32, 50)`. It doesn't change what the RNN computes, only how the tensor's dimensions are arranged.

### <br>6. Compiling and Training the Model
We compile the model using the categorical_crossentropy loss and train it for 500 epochs (lr=0.01) so it has enough iterations to fit this small dataset well.

In [69]:
epochs = 500
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(epochs):
    optimizer.zero_grad()

    y_pred = model(X_one_hot)
    loss = -(y_one_hot * torch.log(y_pred + 1e-9)).sum(dim=1).mean()  # categorical_crossentropy
    loss.backward()
    optimizer.step()

    accuracy = (y_pred.argmax(dim=1) == y_one_hot.argmax(dim=1)).float().mean()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs}, loss: {loss.item():.4f}, accuracy: {accuracy.item():.4f}")

Epoch 50/500, loss: 1.8041, accuracy: 0.4637
Epoch 100/500, loss: 0.3302, accuracy: 0.9437
Epoch 150/500, loss: 0.0272, accuracy: 0.9948
Epoch 200/500, loss: 0.0157, accuracy: 0.9948
Epoch 250/500, loss: 0.0124, accuracy: 0.9948
Epoch 300/500, loss: 0.0109, accuracy: 0.9948
Epoch 350/500, loss: 0.0100, accuracy: 0.9948
Epoch 400/500, loss: 0.0094, accuracy: 0.9948
Epoch 450/500, loss: 0.0091, accuracy: 0.9948
Epoch 500/500, loss: 0.0088, accuracy: 0.9948


### What this code does

This is the **training loop** — it repeats `epochs` (500) times, and each pass does:

1. **`optimizer.zero_grad()`** — clear gradients left over from the previous epoch.
2. **`y_pred = model(X_one_hot)`** — run the model on all training sequences to get predicted next-character probabilities.
3. **`loss = ...`** — measure how wrong `y_pred` is compared to the true `y_one_hot` (categorical cross-entropy: lower = better).
4. **`loss.backward()`** — compute gradients (how much each weight contributed to the error).
5. **`optimizer.step()`** — update the model's weights using those gradients (Adam, `lr=0.01`).
6. **`accuracy = ...`** — % of examples where the model's top prediction matches the true next character.
7. Every 50 epochs, print the current loss and accuracy so we can watch the model improve.

In short: predict → measure error → adjust weights → repeat, so the model gradually gets better at guessing the next character.

### <br>7. Generating New Text Using the Trained Model
After training we use a starting sequence to generate new text character by character.

In [79]:
start_seq = "The soul recovers"
generated_text = start_seq

model.eval()
with torch.no_grad():
    for i in range(113):
        x = torch.tensor([[char_to_index[char] for char in generated_text[-seq_length:]]], dtype=torch.long)
        x_one_hot = torch.nn.functional.one_hot(x, num_classes=len(chars)).float()
        prediction = model(x_one_hot)
        next_index = torch.argmax(prediction, dim=1).item()
        next_char = index_to_char[next_index]
        generated_text += next_char

print("Generated Text:")
print(generated_text)

Generated Text:
The soul recovers radical innocence and beauty born?
Ceremony's a name for the rich horn,
And custom for the spreading laurel tree


### What this code does

- **`model.eval()`** — switches the model to inference mode (turns off training-only behaviors like dropout/batchnorm updates).
- **`with torch.no_grad():`** — tells PyTorch not to track gradients, since we're only doing predictions, not training. This makes generation faster and uses less memory.
- **The loop (50 times):**
  1. Take the **last `seq_length` characters** of `generated_text` and convert them to integer IDs, then to one-hot vectors (`x_one_hot`) — same encoding used during training.
  2. **`model(x_one_hot)`** — run the model to get predicted probabilities for the next character.
  3. **`torch.argmax(prediction, dim=1)`** — pick the character with the highest probability.
  4. **`index_to_char[next_index]`** — convert that prediction back to a character and append it to `generated_text`.
- After 50 iterations, `generated_text` contains the original seed plus 50 newly generated characters.

In short: repeatedly look at the last 3 characters generated so far, predict the next one, append it, and repeat.